In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader,TensorDataset
import torchvision.transforms as transforms

In [2]:
transform=transforms.Compose([
    transforms.ToTensor(),#Convert the Images into pytoch tensor form
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
#Access the Dataset
trainset=CIFAR10(root="./data",train=True,download=True,transform=transform)
testset=CIFAR10(root="./data",train=False,download=True,transform=transform)

100%|██████████| 170M/170M [00:09<00:00, 18.6MB/s]


In [3]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

Build CNN Model


In [5]:
from torch.nn.modules.pooling import MaxPool2d
from torch.nn.modules.activation import ReLU
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers=nn.Sequential(
        #1st function
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),


        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2) ,

    )
    self.fc_layers=nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),

        nn.Linear(256,10)

    )

  def foreward(self,x):
    x=self.conv_layers(x)
    x=x.view(x.size(0),-1) #flatting
    x=self.fc_layers(x)
    return x

In [6]:
model=CNN()
criterian=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

Train CNN


In [7]:
epochs=10

for epoch in range(epochs):
  epoch_train_loss=0.0
  for images,labels in trainloader:
    optimizer.zero_grad()

    output=model.foreward(images)
    loss=criterian(output,labels)
    loss.backward() #Backward
    optimizer.step() #Update W
    epoch_train_loss+=loss.item()
  print(f"for epoch={epoch+1} loss={epoch_train_loss/len(trainloader)}")

for epoch=1 loss=1.3762425486846348
for epoch=2 loss=0.9454022118502565
for epoch=3 loss=0.7520978988131599
for epoch=4 loss=0.6141695887460124
for epoch=5 loss=0.505464397637588
for epoch=6 loss=0.406331037030653
for epoch=7 loss=0.3186199612572522
for epoch=8 loss=0.241721098132603
for epoch=9 loss=0.18253525421309197
for epoch=10 loss=0.14981667982870497


In [9]:
corr=0
total=0
with torch.no_grad():
  for images,label in testloader:
    output=model.foreward(images)
    _,pred=torch.max(output,1)
    corr+=(pred==label).sum().item()
    total+=label.size(0)
  print("Acccuracy=",(corr/total)*100)

Acccuracy= 75.03999999999999
